# Capstone: end-to-end credit portfolio workflow

**Prerequisites:** complete the finstack curriculum **notebooks 01–15**. You should be fluent with core types, market data, instrument pricing, portfolio construction, scenarios, P&L attribution, financial statement modeling, performance analytics, and risk analytics.

This notebook ties every layer into one linear story: **market snapshot → price credit instruments → compute risk metrics → aggregate a portfolio → stress the market → decompose P&L → model an issuer → measure performance → explain model outputs for reporting**.


## Concept: an integrated credit portfolio workflow

Real credit desks rarely use a single API in isolation. A typical internal workflow:

1. **Market setup** — Build a `MarketContext` with discount curves, forward curves, **hazard curves**, and FX for a single as-of date.
2. **Credit instrument pricing** — Price a corporate bond (with `credit_curve_id`), a CDS, and a term loan via `price_instrument_with_metrics`.
3. **Risk metrics** — Extract DV01, CS01, modified duration, and spread measures per instrument.
4. **Portfolio construction** — Aggregate positions across two entities with `value_portfolio`.
5. **Scenario revaluation** — Apply a +50 bp credit stress to hazard curves and compare base vs stressed P&L.
6. **P&L attribution** — Decompose a one-day MTM change into carry, rates, and credit factors using waterfall attribution.
7. **Issuer modeling** — Build a simple P&L bridge (revenue → EBITDA → pre-tax) for credit surveillance.
8. **Performance & risk analytics** — *(disconnected demo)* — Synthetic price-path analysis for CAGR, Sharpe, drawdowns, beta, and ruin probability.
9. **Reporting** — Trace dependencies and explain formulas for a modeled line item.

The code below is **self-contained** (no external data files). Run cells **top to bottom**.


## Step 1 — Market setup

Construct a **`MarketContext`** with:
- **USD OIS** discount curve
- **USD SOFR** forward curve (for floating-rate instruments)
- **ACME-HAZARD** — investment-grade hazard curve (entity: ACME Corp)
- **GLOBEX-HAZARD** — high-yield hazard curve (entity: GLOBEX Inc)
- **EUR/USD** spot via `FxMatrix`

Serialize with `to_json()` so all downstream calls share one market snapshot.


In [ ]:
import json
from datetime import date

from finstack.core.currency import Currency
from finstack.core.market_data import (
    DiscountCurve,
    ForwardCurve,
    FxMatrix,
    HazardCurve,
    MarketContext,
)

AS_OF = date(2025, 1, 15)
AS_OF_STR = AS_OF.isoformat()

mc = MarketContext()

ois = DiscountCurve(
    "USD-OIS",
    AS_OF,
    [(0.0, 1.0), (0.25, 0.9875), (0.5, 0.975), (1.0, 0.95),
     (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    day_count="act_365f",
)
mc.insert(ois)

sofr = ForwardCurve(
    "USD-SOFR",
    0.25,
    [(0.0, 0.045), (1.0, 0.048), (2.0, 0.05), (5.0, 0.052)],
    base_date=AS_OF,
    day_count="act_360",
)
mc.insert(sofr)

acme_hazard = HazardCurve(
    "ACME-HAZARD",
    AS_OF,
    [(0.5, 0.010), (1.0, 0.012), (2.0, 0.015),
     (3.0, 0.018), (5.0, 0.022), (10.0, 0.028)],
    recovery_rate=0.40,
)
mc.insert(acme_hazard)

globex_hazard = HazardCurve(
    "GLOBEX-HAZARD",
    AS_OF,
    [(0.5, 0.035), (1.0, 0.040), (2.0, 0.048),
     (3.0, 0.055), (5.0, 0.062), (10.0, 0.070)],
    recovery_rate=0.35,
)
mc.insert(globex_hazard)

fx = FxMatrix()
fx.set_quote(Currency("EUR"), Currency("USD"), 1.08)
mc.insert_fx(fx)

market_json = mc.to_json()
print(f"MarketContext JSON: {len(market_json):,} chars")
print(f"As-of: {AS_OF_STR}")
print(f"ACME  5Y survival: {acme_hazard.survival(5.0):.6f}  (IG)")
print(f"GLOBEX 5Y survival: {globex_hazard.survival(5.0):.6f}  (HY)")


## Step 2 — Credit instrument pricing

Three credit instruments across two entities:

| Instrument | Entity | Type | Model |
|---|---|---|---|
| **ACME-CORP-5Y** | ACME | Fixed-coupon corporate bond with `credit_curve_id` | `discounting` |
| **CDS-GLOBEX-5Y** | GLOBEX | 5Y CDS (protection buyer) | `hazard_rate` |
| **ACME-TL-B** | ACME | Term loan (fixed spread, amortizing) | `discounting` |


In [ ]:
from finstack.valuations import ValuationResult, price_instrument_with_metrics, validate_instrument_json

acme_bond = json.dumps({
    "type": "bond",
    "spec": {
        "id": "ACME-CORP-5Y",
        "notional": {"amount": "1000000", "currency": "USD"},
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "credit_curve_id": "ACME-HAZARD",
        "cashflow_spec": {
            "Fixed": {
                "coupon_type": "Cash",
                "rate": 0.05,
                "freq": {"count": 6, "unit": "months"},
                "dc": "Thirty360",
                "bdc": "following",
                "calendar_id": "weekends_only",
                "stub": "None",
                "end_of_month": False,
                "payment_lag_days": 0,
            }
        },
        "call_put": None,
        "attributes": {"tags": ["credit"], "meta": {"sector": "Industrials"}},
        "pricing_overrides": {},
    },
})

globex_cds = json.dumps({
    "type": "credit_default_swap",
    "spec": {
        "id": "CDS-GLOBEX-5Y",
        "notional": {"amount": 5000000.0, "currency": "USD"},
        "side": "pay",
        "convention": "isda_na",
        "premium": {
            "start": "2025-03-20",
            "end": "2030-03-20",
            "frequency": {"count": 3, "unit": "months"},
            "stub": "ShortFront",
            "bdc": "following",
            "calendar_id": None,
            "day_count": "Act360",
            "spread_bp": 250.0,
            "discount_curve_id": "USD-OIS",
        },
        "protection": {
            "credit_curve_id": "GLOBEX-HAZARD",
            "recovery_rate": 0.35,
            "settlement_delay": 3,
        },
        "pricing_overrides": {},
        "attributes": {"tags": ["credit"], "meta": {"sector": "Technology"}},
    },
})

acme_loan = json.dumps({
    "type": "term_loan",
    "spec": {
        "id": "ACME-TL-B",
        "notional_limit": {"amount": "10000000", "currency": "USD"},
        "currency": "USD",
        "issue_date": "2024-01-15",
        "maturity": "2029-01-15",
        "discount_curve_id": "USD-OIS",
        "rate": {"Fixed": {"rate_bp": 550}},
        "day_count": "Act360",
        "frequency": {"count": 3, "unit": "months"},
        "bdc": "modified_following",
        "calendar_id": None,
        "stub": "None",
        "amortization": {"PercentPerPeriod": {"bp": 250}},
        "coupon_type": "Cash",
        "settlement_days": 2,
        "attributes": {"tags": ["credit", "loan"], "meta": {"sector": "Industrials"}},
        "pricing_overrides": {},
    },
})

instruments = [
    ("ACME bond", acme_bond, "discounting"),
    ("GLOBEX CDS", globex_cds, "hazard_rate"),
    ("ACME term loan", acme_loan, "discounting"),
]
for label, inst_json, model in instruments:
    validate_instrument_json(inst_json)
    out = price_instrument_with_metrics(inst_json, market_json, AS_OF_STR, model=model)
    vr = ValuationResult.from_json(out)
    print(f"{label:<16} PV = {vr.price:>14,.2f} {vr.currency}")


## Step 3 — Risk metrics

Request DV01, CS01, modified duration, and spread measures per instrument. The metrics registry returns `None` for metrics that don't apply to a given instrument type.


In [ ]:
metric_requests = {
    "ACME bond": (acme_bond, "discounting", ["dv01", "duration_mod", "convexity", "z_spread", "cs01"]),
    "GLOBEX CDS": (globex_cds, "hazard_rate", ["cs01", "cs01_hazard", "jump_to_default"]),
    "ACME term loan": (acme_loan, "discounting", ["dv01", "effective_rate"]),
}

for label, (inst_json, model, mets) in metric_requests.items():
    out = price_instrument_with_metrics(
        inst_json, market_json, AS_OF_STR, model=model, metrics=mets,
    )
    vr = ValuationResult.from_json(out)
    print(f"— {label}")
    for m in mets:
        v = vr.get_metric(m)
        if v is not None:
            print(f"    {m}: {v}")
    print()


## Step 4 — Portfolio construction

Aggregate the three credit positions across two entities (ACME, GLOBEX) using `value_portfolio`. The valuation JSON reports `total_base_ccy` and a per-entity breakdown.


In [ ]:
from finstack.portfolio import value_portfolio

portfolio = json.dumps({
    "id": "credit-portfolio-2025",
    "as_of": AS_OF_STR,
    "base_ccy": "USD",
    "entities": {
        "ACME": {"id": "ACME"},
        "GLOBEX": {"id": "GLOBEX"},
    },
    "positions": [
        {
            "position_id": "POS-ACME-BOND",
            "entity_id": "ACME",
            "instrument_id": "ACME-CORP-5Y",
            "instrument_spec": json.loads(acme_bond),
            "quantity": 1.0,
            "unit": "units",
        },
        {
            "position_id": "POS-GLOBEX-CDS",
            "entity_id": "GLOBEX",
            "instrument_id": "CDS-GLOBEX-5Y",
            "instrument_spec": json.loads(globex_cds),
            "quantity": 1.0,
            "unit": "units",
        },
        {
            "position_id": "POS-ACME-TL",
            "entity_id": "ACME",
            "instrument_id": "ACME-TL-B",
            "instrument_spec": json.loads(acme_loan),
            "quantity": 1.0,
            "unit": "units",
        },
    ],
})

val_json = value_portfolio(portfolio, market_json)
val_obj = json.loads(val_json)
total_pv = float(val_obj["total_base_ccy"]["amount"])
print(f"Portfolio total PV: {total_pv:,.2f} USD\n")
print("By entity:")
for eid, money in val_obj.get("by_entity", {}).items():
    print(f"  {eid}: {float(money['amount']):>14,.2f} {money['currency']}")
print(f"\nPositions valued: {len(val_obj['position_values'])}")


## Step 5 — Scenario revaluation

Apply a **+50 bp par-CDS shock** to both hazard curves and reprice every instrument. The `par_cds` curve kind bumps quoted CDS spreads, which the engine maps into hazard space using the discount curve as anchor. Compare base vs stressed PV to quantify the credit-widening impact.


In [ ]:
from finstack.scenarios import (
    apply_scenario_to_market,
    build_scenario_spec,
    list_builtin_templates,
)
from finstack.valuations import price_instrument

print(f"Built-in templates: {list_builtin_templates()}\n")

stress_ops = json.dumps([
    {
        "kind": "curve_parallel_bp",
        "curve_kind": "par_cds",
        "curve_id": "ACME-HAZARD",
        "discount_curve_id": "USD-OIS",
        "bp": 50.0,
    },
    {
        "kind": "curve_parallel_bp",
        "curve_kind": "par_cds",
        "curve_id": "GLOBEX-HAZARD",
        "discount_curve_id": "USD-OIS",
        "bp": 50.0,
    },
])
scenario_json = build_scenario_spec(
    "credit-stress-50bp",
    stress_ops,
    name="Credit widening +50bp par-CDS",
)
stressed = apply_scenario_to_market(scenario_json, market_json, AS_OF_STR)
stressed_market = stressed["market_json"]
print(f"Applied {stressed['operations_applied']} scenario operations")
if stressed.get("warnings"):
    print("Warnings:", stressed["warnings"])

print(f"\n{'Instrument':<16} {'Base PV':>14} {'Stressed PV':>14} {'Δ P&L':>12}")
print("-" * 60)
for label, inst_json, model in instruments:
    base_out = price_instrument(inst_json, market_json, AS_OF_STR, model=model)
    stress_out = price_instrument(inst_json, stressed_market, AS_OF_STR, model=model)
    pv_base = float(json.loads(base_out)["value"]["amount"])
    pv_stress = float(json.loads(stress_out)["value"]["amount"])
    delta = pv_stress - pv_base
    print(f"{label:<16} {pv_base:>14,.2f} {pv_stress:>14,.2f} {delta:>+12,.2f}")


## Step 6 — P&L attribution

Decompose a one-day MTM change on the ACME bond into **carry**, **rates**, and **credit** factors using **waterfall** attribution. Between T₀ and T₁:

- Discount rates shift **+5 bp**
- Credit spreads (hazard) widen **+10 bp**
- One day of carry accrues


In [ ]:
from finstack.valuations import PnlAttribution, attribute_pnl

AS_OF_T0 = "2025-01-15"
AS_OF_T1 = "2025-01-16"
base_t0 = date.fromisoformat(AS_OF_T0)
base_t1 = date.fromisoformat(AS_OF_T1)

mc_t0 = MarketContext()
mc_t0.insert(DiscountCurve(
    "USD-OIS", base_t0,
    [(0.0, 1.0), (0.25, 0.9875), (0.5, 0.975), (1.0, 0.95),
     (2.0, 0.90), (5.0, 0.75), (10.0, 0.50)],
    day_count="act_365f",
))
mc_t0.insert(HazardCurve(
    "ACME-HAZARD", base_t0,
    [(0.5, 0.010), (1.0, 0.012), (2.0, 0.015),
     (3.0, 0.018), (5.0, 0.022), (10.0, 0.028)],
    recovery_rate=0.40,
))
market_t0 = mc_t0.to_json()

mc_t1 = MarketContext()
mc_t1.insert(DiscountCurve(
    "USD-OIS", base_t1,
    [(0.0, 1.0), (0.25, 0.9872), (0.5, 0.9744), (1.0, 0.9489),
     (2.0, 0.8983), (5.0, 0.7468), (10.0, 0.4963)],
    day_count="act_365f",
))
mc_t1.insert(HazardCurve(
    "ACME-HAZARD", base_t1,
    [(0.5, 0.011), (1.0, 0.013), (2.0, 0.016),
     (3.0, 0.019), (5.0, 0.023), (10.0, 0.029)],
    recovery_rate=0.40,
))
market_t1 = mc_t1.to_json()

waterfall_method = {"Waterfall": ["Carry", "RatesCurves", "CreditCurves"]}
attr_json = attribute_pnl(acme_bond, market_t0, market_t1, AS_OF_T0, AS_OF_T1, waterfall_method)
attr = PnlAttribution.from_json(attr_json)

print(attr.explain())
print(f"\nResidual within tolerance: {attr.residual_within_tolerance()}")
print(f"Repricings: {attr.num_repricings}")


## Step 7 — Issuer modeling

Build a minimal **ACME** P&L spec: explicit drivers (`revenue`, `cogs`, `opex`, `interest`) and computed nodes (`gross_profit`, `ebitda`, `ebt`). `Evaluator` produces a `StatementResult`; `to_pandas_wide()` scans all line items across quarters.


In [ ]:
from finstack.statements import Evaluator, ModelBuilder

b = ModelBuilder("acme-pl")
b.periods("2025Q1..Q4", "2025Q2")
b.value("revenue", [("2025Q1", 250.0), ("2025Q2", 260.0), ("2025Q3", 270.0), ("2025Q4", 280.0)])
b.value("cogs", [("2025Q1", 150.0), ("2025Q2", 155.0), ("2025Q3", 160.0), ("2025Q4", 165.0)])
b.compute("gross_profit", "revenue - cogs")
b.value("opex", [("2025Q1", 50.0), ("2025Q2", 52.0), ("2025Q3", 54.0), ("2025Q4", 56.0)])
b.compute("ebitda", "gross_profit - opex")
b.value("interest", [("2025Q1", 10.0), ("2025Q2", 10.0), ("2025Q3", 10.0), ("2025Q4", 10.0)])
b.compute("ebt", "ebitda - interest")
spec = b.build()
result = Evaluator().evaluate(spec)
wide = result.to_pandas_wide()
print(wide)


## Step 8 — Performance & risk analytics *(disconnected demo)*

> **Note:** This section uses a **synthetic random price path** — it is *not* connected to the portfolio instruments above. It demonstrates the `Performance` and risk analytics APIs in isolation.

Simulate a 252-day equity path, compute CAGR / Sharpe / drawdown, then estimate factor beta and ruin probability.


In [ ]:
import random
from datetime import timedelta

from finstack.analytics import Performance

random.seed(42)
dates = [date(2024, 1, 1) + timedelta(days=i) for i in range(252)]
prices = [100.0]
bench_prices = [100.0]
for _ in range(251):
    prices.append(prices[-1] * (1 + random.gauss(0.0003, 0.012)))
    bench_prices.append(bench_prices[-1] * (1 + random.gauss(0.0004, 0.01)))

perf = Performance.from_arrays(
    dates,
    [prices, bench_prices],
    ["PORTFOLIO", "BENCH"],
    benchmark_ticker="BENCH",
)
print(f"CAGR:    {perf.cagr()[0]:.6f}")
print(f"Sharpe:  {perf.sharpe()[0]:.6f}")
print(f"Sortino: {perf.sortino()[0]:.6f}")
print(f"Ann vol: {perf.volatility()[0]:.6f}")
print(f"VaR 95%: {perf.value_at_risk(0.95)[0]:.6f}")
print(f"ES  95%: {perf.expected_shortfall(0.95)[0]:.6f}")
print(f"Max DD:  {perf.max_drawdown()[0]:.6f}")
print(f"\nBeta vs benchmark: {perf.beta()[0].beta:.4f}")


## Step 9 — Reporting

Export the model spec and evaluation result to JSON, then use `trace_dependencies` for a dependency tree and `explain_formula` for a period-specific walkthrough of how **EBITDA** is computed.


In [ ]:
from finstack.statements_analytics import explain_formula, trace_dependencies

model_json = spec.to_json()
eval_json = result.to_json()

deps = trace_dependencies(model_json, "ebitda")
print("EBITDA dependencies:")
print(deps)

explanation = explain_formula(model_json, eval_json, "ebitda", "2025Q1")
print("\nFormula explanation (2025Q1):")
print(explanation)

## Summary

You walked the full credit portfolio stack:

1. **Market data** — Discount, forward, and **hazard** curves shared by all pricers.
2. **Credit instruments** — Bond (with `credit_curve_id`), CDS, and term loan across two entities.
3. **Risk metrics** — DV01, CS01, duration, z-spread per instrument.
4. **Portfolio** — Entity-level aggregation via `value_portfolio`.
5. **Scenarios** — Par-CDS stress (+50 bp) with base vs stressed revaluation.
6. **P&L attribution** — Waterfall decomposition into carry, rates, and credit.
7. **Issuer modeling** — P&L bridge for credit surveillance.
8. **Performance & risk** — CAGR, Sharpe, beta, ruin probability (synthetic demo).
9. **Reporting** — Dependency tracing and formula explanation.

From here, extend with calibrated curves, more entities, real position data, and wire scenario results into limits and reporting — using the same JSON-first contracts throughout.